# Baseline Model — Car Price Prediction

আগের lesson-এ আমরা `train_linear_regression(X, y)` দিয়ে weights বের করতে শিখেছি। এখন সেই function-কে real car-price dataset-এ ব্যবহার করে আমাদের প্রথম **baseline model** বানাব।

### Baseline কী?

একটি deliberately simple model, যেটা পরে আরও ভালো model-এর সাথে compare করার reference point হিসেবে কাজ করবে।

পুরো flow:

```text
df_train → numerical features → missing values handle → train → predict → compare
```

# 1. Numerical Features Select করা

আমাদের current Linear Regression implementation numerical data নিয়ে কাজ করে। তাই baseline-এর জন্য:

```python
base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']
```

এগুলো থেকে `X_train` তৈরি করব।

- `engine_hp` → horsepower
- `engine_cylinders` → number of cylinders
- `highway_mpg` → highway fuel efficiency
- `city_mpg` → city fuel efficiency
- `popularity` → popularity score

অর্থাৎ `X_train` হলো model-এর input features।

In [ ]:
# See available columns
df_train.columns

In [ ]:
base = [
    'engine_hp',
    'engine_cylinders',
    'highway_mpg',
    'city_mpg',
    'popularity'
]

X_train = df_train[base].values

print("X_train shape:", X_train.shape)
print(X_train[:5])

# 2. Missing Values কেন Problem?

প্রথমে সরাসরি train করলে:

```python
w0, w = train_linear_regression(X_train, y_train)
```

`NaN` থাকলে matrix calculation-এ সেটি ছড়িয়ে গিয়ে weights-ও `NaN` হতে পারে।

যেমন:

$$
NaN\times w = NaN
$$

তাই আগে missing values check করতে হবে।

In [ ]:
# Check missing values in the selected features
df_train[base].isnull().sum()

# 3. Missing Values `0` দিয়ে Fill করা

Baseline-এর জন্য:

```python
X_train = df_train[base].fillna(0).values
```

### কেন `0` কাজ করতে পারে?

ধরি:

$$
g(x_i)=w_0+x_{i1}w_1+x_{i2}w_2
$$

প্রথম feature missing হলে সেটাকে `0` দিলে:

$$
g(x_i)=w_0+0\times w_1+x_{i2}w_2
$$

এবং:

$$
0\times w_1=0
$$

অর্থাৎ ওই feature-এর contribution prediction থেকে disappear করে—model effectively সেটি ignore করে।

**Note:** `0` সবসময় best imputation নয়। Mean/median অনেক ক্ষেত্রে বেশি sensible হতে পারে। কিন্তু baseline-এ simplicity-এর জন্য `0` ব্যবহার করা হচ্ছে।

In [ ]:
X_train = df_train[base].fillna(0).values

print("Missing values after fill:")
print(df_train[base].fillna(0).isnull().sum())

# 4. Baseline Model Train করা

এখন `X_train`-এ missing values নেই। তাই:

```python
w0, w = train_linear_regression(X_train, y_train)
```

চালিয়ে:

- `w0` → bias / intercept
- `w` → 5টি feature-এর weights

পেতে পারি।

আগের lesson-এর function ভিতরে Normal Equation ব্যবহার করে।

In [ ]:
w0, w = train_linear_regression(X_train, y_train)

print("Bias w0:", w0)
print("Feature weights w:", w)

# 5. Prediction করা

Trained weights পাওয়ার পরে আগের vector-form formula ব্যবহার করি:

$$
\boxed{\hat y=w_0+Xw}
$$

Python:

```python
y_pred = w0 + X_train.dot(w)
```

`X_train.dot(w)` প্রতিটি car-এর feature contribution বের করে, তারপর `w0` bias যোগ হয়।

**Note:** `y_train` যদি log-transformed price হয়, তাহলে predictions-ও log scale-এ থাকবে।

In [ ]:
y_pred = w0 + X_train.dot(w)

print("Number of predictions:", len(y_pred))
print("First 10 predictions:")
print(y_pred[:10])

# 6. Predictions বনাম Actual — Histogram

Model-এর predictions কেমন দেখাচ্ছে সেটা visually দেখতে prediction এবং actual target-এর histogram একসাথে plot করতে পারি:

```python
sns.histplot(y_pred, alpha=0.5, bins=50)
sns.histplot(y_train, alpha=0.5, bins=50)
```

`alpha` transparency control করে, তাই দুইটি histogram overlap করলেও দুটো দেখা যায়।

Prediction distribution এবং actual distribution কাছাকাছি হলে model কিছুটা reasonable হতে পারে। কিন্তু histogram subjective—শুধু এটি দেখে model কতটা ভালো তা objectively বলা যায় না। পরের lesson-এ **RMSE** দিয়ে একটি numerical score নেব।

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(y_pred, alpha=0.5, bins=50, label='predictions')
sns.histplot(y_train, alpha=0.5, bins=50, label='actual')

plt.legend()
plt.xlabel('log(price)')
plt.ylabel('Count')
plt.show()

# Final Workflow

```python
base = ['engine_hp', 'engine_cylinders', 'highway_mpg', 'city_mpg', 'popularity']

X_train = df_train[base].fillna(0).values

w0, w = train_linear_regression(X_train, y_train)

y_pred = w0 + X_train.dot(w)
```

### Mental Model

```text
Select features
      ↓
Handle NaN
      ↓
Train → learn w0, w
      ↓
Predict → ŷ = w0 + Xw
      ↓
Compare prediction vs actual
      ↓
Next: RMSE
```

**Baseline model = একটি simple reference model।**

এর performance পরে improve করা models-এর সাথে compare করব।